<a href="https://colab.research.google.com/github/Raistlinhourglass1/LibraryDBbackend/blob/master/Model_Comparison_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Patient-Level Data Distribution

Our training dataset contains 51,182 heartbeats from 27 patient recordings, including 4,785 premature ventricular contractions (PVCs). The PVCs are not distributed equally across patients. Three patients account for approximately 55.3% of the PVC examples, while six patients account for approximately 80.5%.

This imbalance is important because the machine learning models may learn patterns that are more common among patients who contribute a larger number of abnormal heartbeats. To evaluate whether the models can recognize PVCs in new patients, we separated patients into training, validation, and testing groups so that no patient appears in more than one group.

We will examine both overall model performance and patient-level results to better understand how consistently the models identify abnormal heartbeats.

This notebook will compare the performance of two machine learning models developed to classify normal heartbeats and premature ventricular contractions (PVCs): a Support Vector Machine (SVM) and a Multilayer Perceptron (MLP) neural network.

Both models use the same patient-separated datasets, allowing us to evaluate their ability to classify heartbeats from patients not included in training or validation.

We will measure accuracy, precision, recall, F1 score, specificity, and confusion matrices. We will also examine results by patient to identify differences in model performance across individual ECG recordings.

The positive class is PVC (1), while normal heartbeats are labeled 0. The final test dataset will be evaluated only after both models have been selected using the validation data.
2. Load and verify the testing data
This next cell verifies that your test dataset is ready and creates the identifiers we'll use to match both models' predictions.

In [1]:

import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive

# Connect to Google Drive
drive.mount("/content/drive")

# Location of existing ECG project data
data_folder = Path(
    "/content/drive/MyDrive/ECG-Project/data"
)

# Load held-out ECG test labels
with np.load(data_folder / "test_ecg.npz") as data:
    X_test = data["X"]
    y_test = data["y"]

# Load heartbeat metadata
test_metadata = pd.read_csv(
    data_folder / "test_metadata.csv",
    dtype={"Patient_ID": str, "Record": str}
)

# Verify dataset consistency
assert X_test.shape == (15334, 252)
assert len(y_test) == len(test_metadata)
assert np.array_equal(
    y_test,
    test_metadata["Label"].to_numpy()
)

assert set(np.unique(y_test)) == {0, 1}

# Create a stable row identifier
test_reference = test_metadata[
    ["Record", "Patient_ID", "Annotation_Sample"]
].copy()

test_reference.insert(
    0, "Row_ID",
    np.arange(len(test_reference))
)

test_reference["Actual_Label"] = y_test

# Verify that each heartbeat has a unique identifier
assert not test_reference["Row_ID"].duplicated().any()

# Preview the reference table
display(test_reference.head())

print("Total test heartbeats:", len(y_test))
print("Normal heartbeats:", np.sum(y_test == 0))
print("PVC heartbeats:", np.sum(y_test == 1))
print(
    "Unique test patients:",
    test_reference["Patient_ID"].nunique()
)

print("\nTest reference verified successfully.")


Mounted at /content/drive


,Row_ID,Record,Patient_ID,Annotation_Sample,Actual_Label
0,0,103,103,265,0
1,1,103,103,575,0
2,2,103,103,876,0
3,3,103,103,1180,0
4,4,103,103,1482,0


Total test heartbeats: 15334
Normal heartbeats: 14471
PVC heartbeats: 863
Unique test patients: 9

Test reference verified successfully.


This section documents how the models will be evaluated fairly using the same previously unseen patients.

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Directory for completed model predictions
prediction_folder = Path(
    "/content/drive/MyDrive/ECG-Project/predictions"
)
prediction_folder.mkdir(parents=True, exist_ok=True)

model_files = {
    "SVM": "svm_test_predictions.csv",
    "Neural Network": "nn_test_predictions.csv"
}

def evaluate_model(model_name, filename):

    file_path = prediction_folder / filename

    if not file_path.exists():
        print(f"{model_name}: Waiting for {filename}")
        return None

    predictions = pd.read_csv(file_path)

    # Required prediction columns
    required = {"Row_ID", "Predicted_Label"}

    assert required.issubset(predictions.columns), (
        f"{filename} missing required columns"
    )

    # Verify IDs are valid and unique
    assert predictions["Row_ID"].notna().all()
    assert not predictions["Row_ID"].duplicated().any()

    assert set(predictions["Row_ID"]) == set(
        test_reference["Row_ID"]
    ), "Prediction rows do not match the test set"

    # Match predictions to correct heartbeat labels
    merged = test_reference.merge(
        predictions,
        on="Row_ID",
        how="left",
        validate="one_to_one"
    )

    assert merged["Predicted_Label"].notna().all()
    assert merged["Predicted_Label"].isin([0, 1]).all()

    y_true = merged["Actual_Label"].to_numpy()
    y_pred = merged["Predicted_Label"].to_numpy()

    # Confusion matrix: labels 0=Normal, 1=PVC
    cm = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "Specificity": (
            tn / (tn + fp) if tn + fp else np.nan
        ),
        "True_PVCs": int(tp),
        "Missed_PVCs": int(fn),
        "False_Alarms": int(fp)
    }

    # Display confusion matrix
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Normal", "PVC"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"{model_name} - Confusion Matrix")
    plt.show()

    return results


# Evaluate both models if files exist
results = []

for model_name, filename in model_files.items():

    result = evaluate_model(model_name, filename)

    if result is not None:
        results.append(result)

# Show comparison when predictions are available
if results:

    comparison_df = pd.DataFrame(results)

    print("\nMODEL PERFORMANCE COMPARISON")
    display(comparison_df.round(4))

else:
    print(
        "\nNo prediction files found yet. "
        "Evaluation code is ready."
    )


SVM: Waiting for svm_test_predictions.csv
Neural Network: Waiting for nn_test_predictions.csv

No prediction files found yet. Evaluation code is ready.


Patient-Level Model Evaluation
In addition to overall model performance, we will evaluate how accurately the SVM and neural network classify heartbeats from each individual test patient.

The MIT-BIH dataset contains unequal numbers of normal and PVC heartbeats across patients. Because of this imbalance, overall model accuracy may hide poor performance for certain patients.

We will calculate accuracy, precision, recall, and F1 score separately for each test patient. This will help us determine whether the models perform consistently across different patients or struggle with particular ECG recordings.

These results will also help identify limitations in the models' ability to generalize to previously unseen patients.
Colab Cell 6 — Python code
This code uses your existing test_reference, model_files, and prediction_folder variables.

In [4]:

import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_patients(model_name, filename):

    file_path = prediction_folder / filename

    if not file_path.exists():
        print(f"{model_name}: Waiting for predictions")
        return None

    predictions = pd.read_csv(file_path)

    # Match predictions to the original heartbeat rows
    merged = test_reference.merge(
        predictions,
        on="Row_ID",
        how="left",
        validate="one_to_one"
    )

    # Check that every test heartbeat has one prediction
    assert len(predictions) == len(test_reference)
    assert not predictions["Row_ID"].duplicated().any()
    assert merged["Predicted_Label"].notna().all()
    assert merged["Predicted_Label"].isin([0, 1]).all()

    patient_results = []

    for patient_id, group in merged.groupby("Patient_ID"):

        y_true = group["Actual_Label"].to_numpy()
        y_pred = group["Predicted_Label"].to_numpy()

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).ravel()

        patient_results.append({
            "Model": model_name,
            "Patient_ID": patient_id,
            "Heartbeats": len(group),
            "Actual_PVCs": int(np.sum(y_true == 1)),
            "Detected_PVCs": int(tp),
            "Missed_PVCs": int(fn),
            "False_Alarms": int(fp),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "Recall": (
                tp / (tp + fn) if tp + fn > 0 else np.nan
            ),
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            )
        })

    return pd.DataFrame(patient_results)


# Evaluate patient-level results for both models
all_patient_results = []

for model_name, filename in model_files.items():

    result = evaluate_patients(model_name, filename)

    if result is not None:
        all_patient_results.append(result)

if all_patient_results:

    patient_comparison = pd.concat(
        all_patient_results,
        ignore_index=True
    )

    display(
        patient_comparison.sort_values(
            ["Model", "Patient_ID"]
        ).round(3)
    )

else:
    print(
        "Patient-level evaluation is ready. "
        "Waiting for model predictions."
    )


SVM: Waiting for predictions
Neural Network: Waiting for predictions
Patient-level evaluation is ready. Waiting for model predictions.
